In [ ]:
OPCIJE = {
    # tt_split, random_state, C, kernel, gamma
    # "tt_split": [0.1, 0.2, 0.22, 0.25, 0.29, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5],
    "tt_split": [0.2, 0.22, 0.25, 0.29],
    # "tt_split": np.arange(0.1, 0.5, 0.0001).tolist(),
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}


# "tt_split": np.arange(0.32, 0.34, 0.0001).tolist(),

# SVM - Support Vector Machine - algoritam za klasifikaciju i regresiju

# RandomSearchCV - metoda za pronalaženje najboljih hiperparametara modela
# GridSearchCV - metoda za pronalaženje najboljih hiperparametara modela - isprobava sve kombinacije


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import random
import numpy as np
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [ ]:
# Učitavanje podataka
df = pd.read_csv("iris.csv") 

X = df.drop('species', axis=1)
y = df['species']


In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Ovaj dio koda radi evaluaciju eksperimenata dati u varijabli OPCIJE.
# Za svaki eksperiment, dijeli podatke na trening i test skup, trenira SVM model sa zadanim hiperparametrima, i računa F1 score na test skupu.
# Koristeci GridSearchCV ili RandomSearchCV bi bilo efikasnije, ali ovaj kod demonstrira osnovni pristup evaluacije.

# Najbolji parametri: 
best_params = {
    "tt_split": 0.29,
    "random_state": 0,
    "C": 0.1,
    "kernel": "linear",
    "gamma": "scale"
}

best_score = 0
best_model = None

for tt_split in OPCIJE["tt_split"]:
    for random_state in OPCIJE["random_state"]:
        for C in OPCIJE["C"]:
            for kernel in OPCIJE["kernel"]:
                for gamma in OPCIJE["gamma"]:

                    # Podjela podataka na trening i test skup
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tt_split, random_state=random_state)

                    # Treniranje SVM modela
                    model = SVC(C=C, kernel=kernel, gamma=gamma)
                    model.fit(X_train, y_train)

                    # Predviđanje na test skupu
                    y_pred = model.predict(X_test)

                    # Računanje F1 score
                    score = f1_score(y_test, y_pred, average='weighted')
                    # ovdje moze ici bilo koja funkcija EVALUACIJE, npr. accuracy_score, precision_score, recall_score, itd.

                    # Ispis rezultata za trenutne parametre
                    print(f"tt_split: {tt_split}, random_state: {random_state}, C: {C}, kernel: {kernel}, gamma: {gamma} => F1 Score: {score}")
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            "tt_split": tt_split,
                            "random_state": random_state,
                            "C": C,
                            "kernel": kernel,
                            "gamma": gamma
                        }
                        best_model = model
                    
                    
            

In [ ]:
best_params, best_score


In [ ]:
best_model


# Zadaća

1. Uraditi, odnosno zavrisiti RandomSearch
koji je zapocet u zadnjoj celiji posljednjih vjezbi (0.5 bodova)

In [ ]:
# RandomSearch Implementacija
from sklearn.model_selection import RandomizedSearchCV

# ValueError: Invalid parameter 'tt_split' for estimator SVC(). Valid parameters are: ['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'].

OPCIJE = {
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [True, False],
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    # 'decision_function_shape' moze biti 'ovr' ili 'ovo'
    # 'ovo' (one-vs-one) nije kompatibilan sa break_ties=True
    # kada sklearn pokusa koristiti break_ties=True sa 'ovo', baca ValueError
    # zato koristimo samo 'ovr' (one-vs-rest) koji radi sa svim kombinacijama
    "decision_function_shape": ['ovr'],
    "degree": [3, 4, 5],
    # max_iter definise maksimalan broj iteracija koji SVM solver smije napraviti
    # ako stavimo mali broj (100, 200), solver moze prestati prije nego konvergira
    # to uzrokuje ConvergenceWarning: "Solver terminated early"
    # sklearn preporucuje da se podaci skaliraju (StandardScaler) ili da se poveca max_iter
    # najjednostavnije rjesenje je -1 sto znaci "bez limita" i onda solver radi dok ne konvergira
    "max_iter": [-1, 1000, 2000],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0, 1]
}

random_search = RandomizedSearchCV(
    estimator=SVC(), 
    param_distributions=OPCIJE, 
    n_iter=100, 
    scoring='f1_weighted', 
    random_state=0)

random_search.fit(X, y)
random_search.best_params_, random_search.best_score_



2. Konvertovati pseudokod u Python kod da radi search za
na pocetku spomenu algoritam sismisa (+1 bod)

In [ ]:
# za svaki bat i:                     # prolazimo kroz svakog šišmiša pojedinačno
#                                     # i = indeks jednog šišmiša u populaciji

#     beta = random(0, 1)             # slučajan broj između 0 i 1
#                                     # koristi se da frekvencija bude malo drugačija za svakog šišmiša

#     f[i] = f_min + (f_max - f_min) * beta
#                                     # f[i] = frekvencija i-tog šišmiša
#                                     # frekvencija određuje koliko jako mijenja svoje kretanje
#                                     # nije "pozicija", nego parametar koji utiče na pomak

#     v[i] = v[i] + (x[i] - best) * f[i]
#                                     # v[i] = brzina i-tog šišmiša
#                                     # brzina govori koliko i u kojem smjeru će se pomjeriti
#                                     # x[i] = trenutna pozicija tog šišmiša
#                                     # best = trenutno najbolje rješenje od svih šišmiša
#                                     # ovom formulom šišmiš koriguje svoju brzinu u odnosu na best

#     x_new = x[i] + v[i]
#                                     # x_new = nova kandidatska pozicija za tog jednog šišmiša
#                                     # znači: uzmemo staru poziciju i dodamo brzinu
#                                     # DA — ovo je nova pozicija JEDNOG šišmiša, ovog i-tog

#     ako random(0,1) > pulse_rate[i]:
#         x_new = best + epsilon * average_loudness
#                                     # ponekad šišmiš ne ide običnim pomakom
#                                     # nego skoči blizu trenutno najboljeg rješenja
#                                     # epsilon = mali slučajni broj / slučajan mali pomak
#                                     # average_loudness = prosječna glasnoća svih šišmiša
#                                     # ovo služi za lokalnu pretragu oko najboljeg rješenja

#     x_new = popravi_granice(x_new)
#                                     # ako je nova pozicija izašla van dozvoljenog opsega,
#                                     # vrati je unutar granica problema

#     fitness_new = objective(x_new)
#                                     # izračunaj koliko je dobra nova pozicija
#                                     # objective = funkcija koju minimiziraš ili maksimiziraš

#     ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
#         x[i] = x_new
#                                     # prihvati novu poziciju za tog šišmiša

#         fitness[i] = fitness_new
#                                     # zapamti novu vrijednost funkcije za tog šišmiša

#         loudness[i] = alpha * loudness[i]
#                                     # smanji glasnoću tog šišmiša
#                                     # što iteracije više idu, šišmiš postaje "mirniji"

In [ ]:
import random
import pandas as pd
import numpy as np

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.datasets import load_breast_cancer

# --- Učitavanje dataseta i podjela na trening i test skupove ---
df = load_breast_cancer()
X = df.data
y = df.target

# --- PARAMETRI ALGORITMA ---
N_BATS = 10        # broj šišmiša u populaciji (kandidatnih rješenja)
MAX_GEN = 30       # broj iteracija (koliko puta ćemo ažurirati sve šišmiše)
F_MIN, F_MAX = 0.0, 2.0  # min i max frekvencija (kontroliše veličinu pomaka)
ALPHA = 0.9        # faktor smanjenja glasnoće (šišmiš postaje "mirniji" tokom pretrage)

# --- PRETRAZNI PROSTOR ---
LOWER = [0.01, 0.15]   # donje granice: C >= 0.01, tt_split >= 0.15
UPPER = [100.0, 0.45]  # gornje granice: C <= 100, tt_split <= 0.45

# --- OBJECTIVE FUNKCIJA ---
# Prima poziciju šišmiša [C, tt_split]
# Trenira SVM sa tim parametrima
# Vraćamo grešku jer algoritam MINIMIZIRA - što manji broj = bolje rješenje
def objective(position):
    C_val = position[0]   # prvi parametar = C (regularizacija SVM-a)
    tt = position[1]      # drugi parametar = veličina test skupa
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tt, random_state=42)
    model = SVC(C=C_val, kernel="linear", gamma="scale")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return -f1_score(y_test, y_pred, average="weighted")  # negativan F1 = minimiziramo

# --- INICIJALIZACIJA POPULACIJE ---
# Svaki šišmiš = jedna pozicija [C, tt_split] na slučajnom mjestu unutar granica
bats = [[random.uniform(LOWER[j], UPPER[j]) for j in range(2)] for _ in range(N_BATS)]

# Brzine svih šišmiša - na početku nula (šišmiši miruju)
velocities = [[0.0, 0.0] for _ in range(N_BATS)]

# Frekvencije svih šišmiša - na početku nula
frequencies = [0.0] * N_BATS

# Glasnoća svih šišmiša - na početku 1.0 (maksimalna)
# Veća glasnoća = veća šansa da prihvati novo rješenje
loudness = [1.0] * N_BATS

# Puls svih šišmiša - na početku 0.5
# Veći puls = manja šansa za lokalnu pretragu oko najboljeg rješenja
pulse_rate = [0.5] * N_BATS

# Izračunaj fitness (grešku) za svaku početnu poziciju
fitness = [objective(b) for b in bats]

# Pronađi najboljeg šišmiša - onog sa najmanjom greškom (min jer minimiziramo)
best = bats[fitness.index(min(fitness))][:]
best_fit = min(fitness)

# --- GLAVNA PETLJA ---
for t in range(MAX_GEN):
    # Prosječna glasnoća svih šišmiša u ovoj iteraciji
    average_loudness = sum(loudness) / N_BATS

    for i in range(N_BATS):  # prolazimo kroz svakog šišmiša
        # 1) Nasumična frekvencija za ovog šišmiša
        beta = random.random()
        frequencies[i] = F_MIN + (F_MAX - F_MIN) * beta

        # 2) Ažuriraj brzinu - šišmiš ubrzava prema trenutno najboljem rješenju
        velocities[i][0] += (bats[i][0] - best[0]) * frequencies[i]
        velocities[i][1] += (bats[i][1] - best[1]) * frequencies[i]

        # 3) Nova kandidatna pozicija = stara pozicija + brzina
        x_new = [bats[i][0] + velocities[i][0],
                 bats[i][1] + velocities[i][1]]

        # 4) Lokalna pretraga - ponekad šišmiš skoči blizu najboljeg rješenja
        if random.random() > pulse_rate[i]:
            eps = random.uniform(-1, 1)  # mali slučajni pomak
            x_new = [best[j] + eps * average_loudness for j in range(2)]

        # 5) Popravi granice - ako je šišmiš izašao van dozvoljenog prostora, vrati ga
        x_new = np.clip(x_new, LOWER, UPPER) 

        # 6) Izračunaj fitness nove pozicije
        fitness_new = objective(x_new)

        # 7) Prihvati novo rješenje ako je BOLJE (manja greška) i uz vjerovatnoću = loudness
        if fitness_new < fitness[i] and random.random() < loudness[i]:
            bats[i] = x_new
            fitness[i] = fitness_new
            loudness[i] *= ALPHA  # smanji glasnoću - šišmiš postaje "mirniji"

        # 8) Ažuriraj globalno najbolje rješenje
        if fitness[i] < best_fit:
            best = bats[i][:]
            best_fit = fitness[i]

    # Ispis rezultata za svaki bat
    print(f"Iteracija {t+1}: Najbolji C: {best[0]:.4f}, Najbolji tt_split: {best[1]:.4f}, F1: {-best_fit:.4f}")

# Finalni rezultat - najbolji parametri i F1
print(f"\nFinalni najbolji C: {best[0]:.4f}")
print(f"Finalni najbolji tt_split: {best[1]:.4f}")
print(f"Finalni najbolji F1: {-best_fit:.4f}")